# 09 — License Tracking End-to-End Test

**Purpose:** Validate the full license-tracking schema against live Supabase data.  
Hand-picks only the **most important** signals assuming **unique key per user**.

## Data Sources
| Table | What It Tells Us |
|---|---|
| `licenses` | Ownership — who holds each key, plan tier, active/expired |
| `license_validation_logs` | Every validate call — activation proof, usage cadence, errors |
| `license_devices` | Device fleet — platform, broker, account, last heartbeat |

## Tracked Metrics (6 core signals)
1. **License Registry** — key → email → plan → status
2. **Activation Check** — first-ever successful validation timestamp
3. **Last Active** — most recent device heartbeat
4. **Usage Cadence** — distinct active days in rolling 30-day window
5. **Device Fleet** — platform / broker / account per license
6. **Error Signals** — failed validations grouped by error code

---

## 0 — Setup & Connection

In [3]:
import sys, os, json

import pandas as pd

from datetime import datetime, timedelta, timezone



# Workspace root (adjust if running in Railway container)

WS_ROOT = os.environ.get(

    "WORKSPACE_ROOT",

    r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge"

)

if WS_ROOT not in sys.path:

    sys.path.insert(0, WS_ROOT)



from dotenv import load_dotenv

load_dotenv(os.path.join(WS_ROOT, ".env"))



from shared.supabase_client import get_supabase

db = get_supabase(use_service_role=True)



# Rolling window

DAYS = 30

SINCE = (datetime.now(timezone.utc) - timedelta(days=DAYS)).isoformat()



# Shared defaults so downstream cells stay safe even if a query returns nothing

df_licenses = pd.DataFrame()

df_merged = pd.DataFrame()

df_fleet = pd.DataFrame()

df_errors = pd.DataFrame()



print("[OK] Connected to Supabase")

print(f"[OK] Window: last {DAYS} days (since {SINCE[:10]})")

[OK] Connected to Supabase
[OK] Window: last 30 days (since 2026-02-20)


## 1 — License Registry
Every active license key → owner email, plan, expiry, device limit.

In [4]:
licenses_raw = (

    db.table("licenses")

    .select("id, license_key, email, plan, max_devices, is_active, created_at, expires_at")

    .order("created_at", desc=False)

    .execute()

).data or []



df_licenses = pd.DataFrame(licenses_raw)



if not df_licenses.empty:

    df_licenses["created_at"] = pd.to_datetime(

        df_licenses["created_at"], format="mixed", utc=True, errors="coerce"

    ).dt.strftime("%Y-%m-%d")

    df_licenses["expires_at"] = pd.to_datetime(

        df_licenses["expires_at"], format="mixed", utc=True, errors="coerce"

    ).dt.strftime("%Y-%m-%d")

    df_licenses["key_short"] = df_licenses["license_key"].str[:12] + "..."

    display_cols = ["key_short", "email", "plan", "max_devices", "is_active", "created_at", "expires_at"]

    print(f"Total licenses: {len(df_licenses)}  |  Active: {int(df_licenses['is_active'].fillna(False).sum())}")

    print()

    print(df_licenses[display_cols].to_string(index=False))

else:

    print("[!] No licenses found")

Total licenses: 6  |  Active: 6

      key_short                    email         plan  max_devices  is_active created_at expires_at
HEDGE-DEV-20... developer@hedge-edge.com   enterprise          100       True 2026-02-11 2030-12-31
LJMEX-TXUQO-...       test@hedgeedge.com professional            3       True 2026-03-12 2027-03-12
ORRDC-V1O1T-...     friend@hedgeedge.com professional            3       True 2026-03-12 2027-12-31
TSZ38-SVQYJ-...  developer@hedgedge.info   enterprise          100       True 2026-03-12 2027-12-31
6WV6C-0JO7D-... developer@hedge-edge.com professional            3       True 2026-03-13 2027-03-12
HEDGE-BETA-F...       beta@hedgedge.info   enterprise          100       True 2026-03-17 2035-12-31


## 2 — Activation Status
First-ever **successful** validation per license = activation timestamp.  
If empty → user received key but **never opened the app**.

In [5]:
# Pull all successful validations

val_logs = (

    db.table("license_validation_logs")

    .select("license_key, created_at")

    .eq("success", True)

    .order("created_at", desc=False)

    .execute()

).data or []



df_val = pd.DataFrame(val_logs)



activation = {}

if not df_val.empty:

    df_val["created_at"] = pd.to_datetime(

        df_val["created_at"], format="mixed", utc=True, errors="coerce"

    )

    first_seen = df_val.groupby("license_key")["created_at"].min().reset_index()

    first_seen.columns = ["license_key", "activated_at"]

    activation = dict(zip(first_seen["license_key"], first_seen["activated_at"]))



# Map back to licenses

if not df_licenses.empty:

    def match_activation(row):

        full_key = row["license_key"]

        trunc_key = full_key[:20] + "..." if len(full_key) > 20 else full_key

        ts = activation.get(trunc_key) or activation.get(full_key)

        return ts.strftime("%Y-%m-%d %H:%M") if pd.notna(ts) else "NEVER"



    df_licenses["activated_at"] = df_licenses.apply(match_activation, axis=1)

    never_count = (df_licenses["activated_at"] == "NEVER").sum()

    print(f"Activated: {len(df_licenses) - never_count} / {len(df_licenses)}")

    print(f"Never opened app: {never_count}")

    print()

    print(df_licenses[["key_short", "email", "activated_at"]].to_string(index=False))

else:

    print("[!] No license data")

Activated: 4 / 6
Never opened app: 2

      key_short                    email     activated_at
HEDGE-DEV-20... developer@hedge-edge.com            NEVER
LJMEX-TXUQO-...       test@hedgeedge.com 2026-03-12 15:43
ORRDC-V1O1T-...     friend@hedgeedge.com 2026-03-12 16:15
TSZ38-SVQYJ-...  developer@hedgedge.info 2026-03-12 19:02
6WV6C-0JO7D-... developer@hedge-edge.com 2026-03-13 06:24
HEDGE-BETA-F...       beta@hedgedge.info            NEVER


## 3 — Last Active (Most Recent Heartbeat)
When was each user last seen? Uses `license_devices.last_seen_at`.  
Stale > 7 days = at-risk churn.

In [6]:
devices_raw = (

    db.table("license_devices")

    .select("license_id, device_id, last_seen_at, is_active")

    .eq("is_active", True)

    .execute()

).data or []



df_devs = pd.DataFrame(devices_raw)

df_merged = pd.DataFrame()



if not df_devs.empty and not df_licenses.empty:

    df_devs["last_seen_at"] = pd.to_datetime(

        df_devs["last_seen_at"], format="mixed", utc=True, errors="coerce"

    )

    last_seen = df_devs.groupby("license_id")["last_seen_at"].max().reset_index()

    last_seen.columns = ["id", "last_active"]



    df_merged = df_licenses.merge(last_seen, on="id", how="left")

    now = pd.Timestamp.now(tz="UTC")

    df_merged["days_since"] = (now - df_merged["last_active"]).dt.days

    df_merged["status"] = df_merged["days_since"].apply(

        lambda d: "ACTIVE" if pd.notna(d) and d <= 3

        else "IDLE" if pd.notna(d) and d <= 7

        else "AT RISK" if pd.notna(d) else "NEVER SEEN"

    )

    df_merged["last_active_str"] = df_merged["last_active"].dt.strftime("%Y-%m-%d %H:%M").fillna("-")



    print(f"Active (<=3d): {(df_merged['status'] == 'ACTIVE').sum()}")

    print(f"Idle (4-7d):  {(df_merged['status'] == 'IDLE').sum()}")

    print(f"At Risk (>7d): {(df_merged['status'] == 'AT RISK').sum()}")

    print(f"Never Seen:   {(df_merged['status'] == 'NEVER SEEN').sum()}")

    print()

    print(df_merged[["key_short", "email", "last_active_str", "days_since", "status"]].to_string(index=False))

else:

    print("[!] No device data")

Active (<=3d): 1
Idle (4-7d):  2
At Risk (>7d): 1
Never Seen:   2

      key_short                    email  last_active_str  days_since     status
HEDGE-DEV-20... developer@hedge-edge.com                -         NaN NEVER SEEN
LJMEX-TXUQO-...       test@hedgeedge.com 2026-03-12 16:37        10.0    AT RISK
ORRDC-V1O1T-...     friend@hedgeedge.com 2026-03-17 22:55         5.0       IDLE
TSZ38-SVQYJ-...  developer@hedgedge.info 2026-03-22 23:46         0.0     ACTIVE
6WV6C-0JO7D-... developer@hedge-edge.com 2026-03-17 22:55         5.0       IDLE
HEDGE-BETA-F...       beta@hedgedge.info                -         NaN NEVER SEEN


## 4 — Usage Cadence (30-Day)
Distinct days with at least one successful validation per license.  
This is the **strongest engagement signal** — shows habitual use vs one-time try.

In [7]:
recent_logs = (
    db.table("license_validation_logs")
    .select("license_key, created_at")
    .eq("success", True)
    .gte("created_at", SINCE)
    .execute()
).data or []

df_recent = pd.DataFrame(recent_logs)

if not df_recent.empty and not df_licenses.empty:
    df_recent["date"] = pd.to_datetime(df_recent["created_at"]).dt.date
    cadence = df_recent.groupby("license_key")["date"].nunique().reset_index()
    cadence.columns = ["license_key_trunc", "days_active"]

    # Map truncated keys back to full licenses
    def get_cadence(row):
        full_key = row["license_key"]
        trunc = full_key[:20] + "..." if len(full_key) > 20 else full_key
        match = cadence[cadence["license_key_trunc"] == trunc]
        if match.empty:
            match = cadence[cadence["license_key_trunc"] == full_key]
        return int(match["days_active"].iloc[0]) if not match.empty else 0

    df_licenses["days_active_30d"] = df_licenses.apply(get_cadence, axis=1)
    df_licenses["engagement"] = df_licenses["days_active_30d"].apply(
        lambda d: "POWER USER" if d >= 15
        else "REGULAR" if d >= 5
        else "LOW" if d >= 1
        else "DORMANT"
    )

    total_events = len(df_recent)
    print(f"Total successful validations (30d): {total_events}")
    print()
    print(df_licenses[["key_short", "email", "days_active_30d", "engagement"]].to_string(index=False))
else:
    print("[!] No recent validation logs")

Total successful validations (30d): 1000

      key_short                    email  days_active_30d engagement
HEDGE-DEV-20... developer@hedge-edge.com                0    DORMANT
LJMEX-TXUQO-...       test@hedgeedge.com                1        LOW
ORRDC-V1O1T-...     friend@hedgeedge.com                5    REGULAR
TSZ38-SVQYJ-...  developer@hedgedge.info                5    REGULAR
6WV6C-0JO7D-... developer@hedge-edge.com                4        LOW
HEDGE-BETA-F...       beta@hedgedge.info                0    DORMANT


## 5 — Device Fleet
Per-license breakdown: platform (MT4/MT5/cTrader), broker, trading account ID.  
Tells you exactly **what** each user is running.

In [8]:
fleet_raw = (

    db.table("license_devices")

    .select("license_id, device_id, platform, broker, account_id, version, is_active, first_seen_at, last_seen_at")

    .order("last_seen_at", desc=True)

    .execute()

).data or []



df_fleet = pd.DataFrame(fleet_raw)



if not df_fleet.empty and not df_licenses.empty:

    license_map = dict(zip(df_licenses["id"], df_licenses["key_short"]))

    email_map = dict(zip(df_licenses["id"], df_licenses["email"]))

    df_fleet["key_short"] = df_fleet["license_id"].map(license_map)

    df_fleet["email"] = df_fleet["license_id"].map(email_map)

    df_fleet["device_short"] = df_fleet["device_id"].str[:16] + "..."

    df_fleet["first_seen_at"] = pd.to_datetime(

        df_fleet["first_seen_at"], format="mixed", utc=True, errors="coerce"

    ).dt.strftime("%Y-%m-%d")

    df_fleet["last_seen_at"] = pd.to_datetime(

        df_fleet["last_seen_at"], format="mixed", utc=True, errors="coerce"

    ).dt.strftime("%Y-%m-%d %H:%M")



    show_cols = ["key_short", "email", "device_short", "platform", "broker", "account_id", "version", "is_active", "last_seen_at"]

    active_fleet = df_fleet[df_fleet["is_active"] == True]

    print(f"Total devices: {len(df_fleet)}  |  Active: {len(active_fleet)}")

    print()

    print(active_fleet[show_cols].to_string(index=False))

else:

    print("[!] No device fleet data")

Total devices: 17  |  Active: 17

      key_short                    email        device_short platform                              broker account_id version  is_active     last_seen_at
TSZ38-SVQYJ-...  developer@hedgedge.info F7F96D1A01404A9B...  unknown                      FundedNext Ltd   13985776   0.0.0       True 2026-03-22 23:46
TSZ38-SVQYJ-...  developer@hedgedge.info F909E31501F64C5C...  unknown Vantage International Group Limited   22641234   0.0.0       True 2026-03-22 23:46
TSZ38-SVQYJ-...  developer@hedgedge.info a8959bca0af749b6...  desktop                                 NaN        NaN   0.0.0       True 2026-03-22 23:26
TSZ38-SVQYJ-...  developer@hedgedge.info 8B56A7D2AB789A10...  unknown                     MetaQuotes Ltd.  104338734   0.0.0       True 2026-03-22 15:50
TSZ38-SVQYJ-...  developer@hedgedge.info ABCDEF0123456789...  unknown                          TestBroker   12345678   0.0.0       True 2026-03-22 03:56
TSZ38-SVQYJ-...  developer@hedgedge.info 29B6184

## 6 — Error Signals
Failed validation attempts grouped by license + error code.  
High error counts = user struggling → proactive support opportunity.

In [9]:
error_logs = (

    db.table("license_validation_logs")

    .select("license_key, error_code, error_message, created_at")

    .eq("success", False)

    .gte("created_at", SINCE)

    .execute()

).data or []



df_errors = pd.DataFrame(error_logs)



if not df_errors.empty:

    error_summary = (

        df_errors.groupby(["license_key", "error_code"])

        .agg(count=("created_at", "size"), last_error=("created_at", "max"))

        .reset_index()

        .sort_values("count", ascending=False)

    )

    error_summary["last_error"] = pd.to_datetime(

        error_summary["last_error"], format="mixed", utc=True, errors="coerce"

    ).dt.strftime("%Y-%m-%d %H:%M")



    total_errors = len(df_errors)

    unique_keys = df_errors["license_key"].nunique()

    print(f"Total failed validations (30d): {total_errors}")

    print(f"Affected license keys: {unique_keys}")

    print()

    print(error_summary.to_string(index=False))

else:

    print("[OK] No failed validations in the last 30 days")

Total failed validations (30d): 31
Affected license keys: 13

            license_key           error_code  count       last_error
TSZ38-SVQYJ-U72O4-QR...   ERROR_DEVICE_LIMIT      5 2026-03-18 14:32
TSZ38-SVQYJ-U72O4-QR... ERROR_CREEM_REJECTED      5 2026-03-17 16:48
TSZ38-SVQYJ-U7204-QR... ERROR_CREEM_REJECTED      4 2026-03-13 01:24
ORRDC-V1O1T-L2J2T-W6... ERROR_CREEM_REJECTED      2 2026-03-17 23:00
6WV6C-0JO7D-Y350G-TC... ERROR_CREEM_REJECTED      2 2026-03-17 23:00
6WV6C-0JO7D-Y350G-TC...   ERROR_DEVICE_LIMIT      2 2026-03-16 21:56
AAAAA-BBBBB-CCCCC-DD... ERROR_CREEM_REJECTED      2 2026-03-12 19:44
UN0HK-52NXO-F6COR-0U... ERROR_CREEM_REJECTED      2 2026-03-12 15:19
LJMEX-TXUQO-F1P7T-79... ERROR_CREEM_REJECTED      1 2026-03-12 15:46
9L6LL-1BE2D-946MM-04... ERROR_CREEM_REJECTED      1 2026-03-12 15:52
   HEDGE-BETA-FREE-2026 ERROR_CREEM_REJECTED      1 2026-03-17 22:47
Q111K-GIMDZ-519SP-HV... ERROR_CREEM_REJECTED      1 2026-03-12 15:16
TT1KED84WB6CZPVXMFR4... ERROR_CREEM_REJEC

## 7 — Customer Health Dashboard
**Single unified view** combining all 6 signals per license.  
This is the table you check to see who's healthy, churning, or stuck.

In [10]:
if not df_licenses.empty:
    health = df_licenses[["key_short", "email", "plan", "is_active", "expires_at", "activated_at"]].copy()

    # Last active + status (from cell 3)
    if "last_active_str" in df_merged.columns:
        health = health.merge(
            df_merged[["key_short", "last_active_str", "days_since", "status"]],
            on="key_short", how="left"
        )
    else:
        health["last_active_str"] = "—"
        health["days_since"] = None
        health["status"] = "UNKNOWN"

    # Usage cadence (from cell 4)
    if "days_active_30d" in df_licenses.columns:
        cadence_map = dict(zip(df_licenses["key_short"], df_licenses["days_active_30d"]))
        engagement_map = dict(zip(df_licenses["key_short"], df_licenses["engagement"]))
        health["days_active_30d"] = health["key_short"].map(cadence_map).fillna(0).astype(int)
        health["engagement"] = health["key_short"].map(engagement_map).fillna("DORMANT")
    else:
        health["days_active_30d"] = 0
        health["engagement"] = "DORMANT"

    # Error count (from cell 6)
    if not df_errors.empty:
        err_per_key = df_errors.groupby("license_key").size().to_dict()
        def get_errors(row):
            full_key = df_licenses.loc[df_licenses["key_short"] == row["key_short"], "license_key"]
            if full_key.empty:
                return 0
            fk = full_key.iloc[0]
            trunc = fk[:20] + "..." if len(fk) > 20 else fk
            return err_per_key.get(trunc, err_per_key.get(fk, 0))
        health["errors_30d"] = health.apply(get_errors, axis=1)
    else:
        health["errors_30d"] = 0

    # Device count
    if not df_fleet.empty:
        active_devs = df_fleet[df_fleet["is_active"] == True].groupby("key_short").size().to_dict()
        health["devices"] = health["key_short"].map(active_devs).fillna(0).astype(int)
    else:
        health["devices"] = 0

    # Overall health score
    def score(row):
        if row["activated_at"] == "NEVER":
            return "NOT ACTIVATED"
        if row["status"] == "AT RISK":
            return "CHURNING"
        if row["errors_30d"] > 10:
            return "NEEDS SUPPORT"
        if row["engagement"] == "POWER USER":
            return "HEALTHY"
        if row["engagement"] == "REGULAR":
            return "HEALTHY"
        if row["engagement"] == "LOW":
            return "WARMING UP"
        return "DORMANT"

    health["health"] = health.apply(score, axis=1)

    final_cols = [
        "email", "plan", "activated_at", "last_active_str",
        "days_active_30d", "devices", "errors_30d", "engagement", "health"
    ]
    print("=" * 120)
    print("CUSTOMER HEALTH DASHBOARD")
    print("=" * 120)
    print()
    print(health[final_cols].to_string(index=False))
    print()
    print("─" * 60)
    print(f"HEALTHY:        {(health['health']=='HEALTHY').sum()}")
    print(f"WARMING UP:     {(health['health']=='WARMING UP').sum()}")
    print(f"NOT ACTIVATED:  {(health['health']=='NOT ACTIVATED').sum()}")
    print(f"NEEDS SUPPORT:  {(health['health']=='NEEDS SUPPORT').sum()}")
    print(f"CHURNING:       {(health['health']=='CHURNING').sum()}")
    print(f"DORMANT:        {(health['health']=='DORMANT').sum()}")
else:
    print("[!] No data to build dashboard")

CUSTOMER HEALTH DASHBOARD

                   email         plan     activated_at  last_active_str  days_active_30d  devices  errors_30d engagement        health
developer@hedge-edge.com   enterprise            NEVER                -                0        0           0    DORMANT NOT ACTIVATED
      test@hedgeedge.com professional 2026-03-12 15:43 2026-03-12 16:37                1        2           1        LOW      CHURNING
    friend@hedgeedge.com professional 2026-03-12 16:15 2026-03-17 22:55                5        2           2    REGULAR       HEALTHY
 developer@hedgedge.info   enterprise 2026-03-12 19:02 2026-03-22 23:46                5       10          10    REGULAR       HEALTHY
developer@hedge-edge.com professional 2026-03-13 06:24 2026-03-17 22:55                4        3           4        LOW    WARMING UP
      beta@hedgedge.info   enterprise            NEVER                -                0        0           1    DORMANT NOT ACTIVATED

───────────────────────────

## Summary

### What each column means for customer service:

| Column | Action |
|---|---|
| `activated_at = NEVER` | User got key but never opened app → send onboarding nudge |
| `status = AT RISK` | No heartbeat >7 days → check if they're stuck |
| `errors_30d > 10` | Repeated failures → proactive support ticket |
| `engagement = POWER USER` | >15 active days → candidate for testimonial/referral |
| `engagement = DORMANT` | 0 days active → re-engagement email |
| `devices > max_devices` | Should never happen (enforced by backend) — flag if it does |

### Pre-requisite for clean data:
Each beta tester must have a **unique license key**. Shared keys pollute every metric above.  
Once unique keys are issued, re-run this notebook to get a clean baseline.